# Setup Build Waves Job

Patches the bundle-deployed build jobs so their Genie Code tasks are real `genie_task`s. Runs as the setup job after **every** `bundle deploy`, because deploy restores the placeholders. Idempotent.

The build job is declared in `build_waves.job.yml` with a single placeholder task. This notebook:

1. **Publishes UC Skills** from `skills_dir/` to `{catalog}.{schema}.{skill_id}` (create, upload via Files API, finalize).
2. **Creates or updates Genie Code automations** (one per role: runner, integrator, smoke).
3. **Reads `waves.json`** and dynamically generates the full task graph: per wave, a `for_each` of session runners at full wave width, then an integrator, then a gate check notebook.
4. **Resets the build job** via `jobs/reset` with the generated task graph (replacing the single placeholder).
5. **Patches the smoke job** the same way (simple 2-item for_each).
6. Optionally **runs the smoke job** and checks the output.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
import os

# Set by setup_build_waves.job.yml; fill build_job_id / smoke_job_id by hand for interactive runs
# (bundle summary shows them after deploy).
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")
dbutils.widgets.text("skills_dir", "", "Skills directory (absolute path)")
dbutils.widgets.text("skills_catalog", "", "UC catalog for skills")
dbutils.widgets.text("skills_schema", "", "UC schema for skills")
dbutils.widgets.text("runner_skill_id", "genie_build_session_runner", "Runner skill id")
dbutils.widgets.text("integrator_skill_id", "genie_build_wave_integrator", "Integrator skill id")
dbutils.widgets.text("build_job_id", "", "Build job id")
dbutils.widgets.text("smoke_job_id", "", "Smoke job id")
dbutils.widgets.text("gate_notebook_path", "", "Gate notebook deployed path")
dbutils.widgets.text("placeholder_path", "", "Placeholder notebook deployed path")
dbutils.widgets.dropdown("create_skills_schema", "true", ["true", "false"], "Auto-create skills schema")
dbutils.widgets.dropdown("run_smoke", "false", ["true", "false"], "Run smoke job after patching")

build_root = dbutils.widgets.get("build_root").rstrip("/")
manifest_path = dbutils.widgets.get("manifest_path").strip()
skills_dir = dbutils.widgets.get("skills_dir").strip()
skills_catalog = dbutils.widgets.get("skills_catalog").strip()
skills_schema = dbutils.widgets.get("skills_schema").strip()
runner_skill_id = dbutils.widgets.get("runner_skill_id").strip()
integrator_skill_id = dbutils.widgets.get("integrator_skill_id").strip()
build_job_id = dbutils.widgets.get("build_job_id").strip()
smoke_job_id = dbutils.widgets.get("smoke_job_id").strip()
gate_notebook_path = dbutils.widgets.get("gate_notebook_path").strip()
placeholder_path = dbutils.widgets.get("placeholder_path").strip()
create_skills_schema = dbutils.widgets.get("create_skills_schema") == "true"
run_smoke = dbutils.widgets.get("run_smoke") == "true"

if not build_job_id or not smoke_job_id:
    raise ValueError(
        "build_job_id and smoke_job_id are required. "
        "Run `databricks bundle summary -o json` to find them after deploy."
    )

# Resolve manifest path: if relative, join with build_root
if not os.path.isabs(manifest_path):
    manifest_path = os.path.join(build_root, manifest_path)

print(f"build_root:      {build_root}")
print(f"manifest_path:   {manifest_path}")
print(f"skills_dir:      {skills_dir}")
print(f"skills:          {skills_catalog}.{skills_schema}")
print(f"build_job_id:    {build_job_id}")
print(f"smoke_job_id:    {smoke_job_id}")

In [0]:
import json

with open(manifest_path) as f:
    manifest = json.load(f)

SKILL_PARENT = f"schemas/{skills_catalog}.{skills_schema}"

# Skills to publish: key -> {skill_id, folder, fqn, files{relpath: text}}
SKILLS = {}
for key, skill_id in [("runner", runner_skill_id), ("integrator", integrator_skill_id)]:
    # Resolve folder: manifest may override, otherwise use skills_dir convention
    if manifest.get("skills", {}).get(key, {}).get("folder"):
        folder = os.path.join(build_root, manifest["skills"][key]["folder"])
    else:
        # Convention: skills_dir/<role-name>/
        role_dir = "genie-build-session-runner" if key == "runner" else "genie-build-wave-integrator"
        folder = os.path.join(skills_dir, role_dir)

    files = {}
    for dirpath, _, filenames in os.walk(folder):
        for fn in filenames:
            full = os.path.join(dirpath, fn)
            with open(full) as fh:
                files[os.path.relpath(full, folder)] = fh.read()
    if "SKILL.md" not in files:
        raise FileNotFoundError(f"{folder}/SKILL.md not found")

    fqn = f"{skills_catalog}.{skills_schema}.{skill_id}"
    SKILLS[key] = {"skill_id": skill_id, "folder": folder, "fqn": fqn, "files": files}
    print(f"  {key}: {fqn} ({len(files)} files from {folder})")

PLACEHOLDER = placeholder_path.rsplit("/", 1)[-1] if placeholder_path else "genie_task_placeholder"

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

schema_fqn = f"{skills_catalog}.{skills_schema}"

if create_skills_schema:
    try:
        w.schemas.get(schema_fqn)
        print(f"  Schema {schema_fqn} already exists")
    except Exception:
        print(f"  Creating schema {schema_fqn} ...")
        w.schemas.create(name=skills_schema, catalog_name=skills_catalog)
        print(f"  Schema {schema_fqn} created")
else:
    # Validate the schema exists even when not auto-creating
    try:
        w.schemas.get(schema_fqn)
        print(f"  Schema {schema_fqn} exists (create_skills_schema=false)")
    except Exception as e:
        raise RuntimeError(
            f"Schema {schema_fqn} does not exist and create_skills_schema is false. "
            f"Either create the schema manually or set create_skills_schema=true. Error: {e}"
        )

In [0]:
import io
from databricks.sdk.errors import NotFound, AlreadyExists
from databricks.sdk.service.catalog import Skill

# w = WorkspaceClient() — already created in the schema cell above
skills_published = {}

for key, sk in SKILLS.items():
    fqn = sk["fqn"]
    skill_name = f"skills/{fqn}"  # get/finalize need the skills/ prefix
    ok = False
    try:
        # 1. Create or verify skill exists (SDK method, not raw API)
        try:
            w.ai_gateway.get_skill(name=skill_name)
            print(f"  {key}: skill {fqn} already exists")
        except NotFound:
            try:
                w.ai_gateway.create_skill(
                    parent=SKILL_PARENT,
                    skill_id=sk["skill_id"],
                    skill=Skill(comment=f"Build wave {key} skill"),  # must be Skill dataclass, not dict
                )
                print(f"  {key}: created skill {fqn}")
            except AlreadyExists:
                print(f"  {key}: skill {fqn} already exists (race)")

        # 2. Upload files via the Files API (no upload_skill method exists)
        for relpath, text in sk["files"].items():
            upload_path = f"/Skills/{skills_catalog}/{skills_schema}/{sk['skill_id']}/{relpath}"
            w.files.upload(upload_path, io.BytesIO(text.encode("utf-8")), overwrite=True)

        # 3. Finalize (must use skills/ prefix in the name)
        w.ai_gateway.finalize_skill(name=skill_name)
        print(f"  {key}: published {fqn} ({len(sk['files'])} files)")
        ok = True
    except Exception as e:
        print(f"  {key}: WARN publish failed ({e}); will use workspace SKILL.md as fallback")
    skills_published[key] = ok

In [0]:
uid = w.current_user.me().id

existing = w.api_client.do(
    "GET",
    "/api/2.0/alerts-internal/scheduled-insights-list/GENIE_CODE",
    query={"parent_asset_name": f"users/{uid}"},
).get("scheduled_insights", [])

# --- Fix for duplicate automations on re-run ---
# The list endpoint may not return display_name, so also check what the build job
# already points to. If the job has genie_task configuration_ids from a prior setup,
# reuse those automations instead of creating new ones.
_job_automation_names = set()
try:
    job_detail = w.api_client.do("GET", f"/api/2.1/jobs/get", query={"job_id": build_job_id})
    for task in job_detail.get("settings", {}).get("tasks", []):
        gt = task.get("genie_task", {}) or {}
        cid = gt.get("configuration_id", "")
        if cid:
            _job_automation_names.add(cid)
        fe = task.get("for_each_task", {}) or {}
        inner_gt = fe.get("task", {}).get("genie_task", {}) or {}
        inner_cid = inner_gt.get("configuration_id", "")
        if inner_cid:
            _job_automation_names.add(inner_cid)
except Exception:
    pass  # first deploy — job may still have placeholder

# Build a lookup from name -> automation for job-referenced automations
_job_automations_by_name = {e["name"]: e for e in existing if e.get("name") in _job_automation_names}


def thin_prompt(role: str, key: str, params: list[str]) -> str:
    """Short automation prompt: load the UC Skill, fall back to the workspace SKILL.md."""
    sk = SKILLS[key]
    source = (
        f"Load and follow the `{sk['fqn']}` UC Skill exactly.\n"
        f"If you cannot load it, read and follow `{sk['folder']}/SKILL.md` instead.\n"
        if skills_published[key]
        else f"Read and follow `{sk['folder']}/SKILL.md` exactly.\n"
    )
    param_lines = "\n".join(f"- {p}" for p in params)
    return f"{source}\nYour parameters (from the job task):\n{param_lines}\n"


# Three automations: runner, integrator, smoke
RUNNER_PARAMS = ["session_id", "wave", "run_id", "branch", "build_root", "manifest_path"]
INTEGRATOR_PARAMS = ["wave", "run_id", "branch", "build_root", "manifest_path"]
SMOKE_PROMPT = (
    "You are a smoke test. Reply with exactly: "
    "'smoke_ok session_id=<session_id> run_id=<run_id>'. Nothing else."
)

automation_configs = [
    ("runner", "runner", thin_prompt("runner", "runner", RUNNER_PARAMS)),
    ("integrator", "integrator", thin_prompt("integrator", "integrator", INTEGRATOR_PARAMS)),
    ("smoke", None, SMOKE_PROMPT),
]

configuration_ids = {}
for auto_key, skill_key, prompt in automation_configs:
    # Find existing by matching the key in the display_name
    display_name = f"genie-build-waves:{auto_key}"
    # Try matching by display_name first, then by name from the job's existing tasks
    match = next((e for e in existing if e.get("display_name") == display_name), None)
    if not match and auto_key in ("runner", "integrator"):
        # Check if the job already references an automation we can reuse
        for name, aut in _job_automations_by_name.items():
            # Match runner/integrator by checking the prompt content
            if auto_key == "runner" and "session_id" in aut.get("user_prompt", aut.get("prompt", "")):
                match = aut
                break
            elif auto_key == "integrator" and "session_id" not in aut.get("user_prompt", aut.get("prompt", "")) and "wave" in aut.get("user_prompt", aut.get("prompt", "")):
                match = aut
                break

    if match:
        # Update via PATCH with etag (optimistic concurrency)
        iid = match["id"]
        full_name = match["name"]
        etag = match.get("etag") or w.api_client.do(
            "GET",
            f"/api/2.0/alerts-internal/scheduled-insights/{iid}",
            query={"name": full_name},
        )["etag"]
        w.api_client.do("PATCH", f"/api/2.0/alerts-internal/scheduled-insights/{iid}", body={
            "insight_id": iid,
            "update_mask": "user_prompt,display_name",
            "etag": etag,
            "scheduled_insight": {
                "name": full_name,
                "display_name": display_name,
                "user_prompt": prompt,
            },
        })
        # genie_task.configuration_id expects the full name, not just the hex ID
        configuration_ids[auto_key] = full_name
        print(f"  {auto_key}: updated automation {full_name}")
    else:
        # Create via POST — parent_asset_name at top level, rest in scheduled_insight
        resp = w.api_client.do("POST", "/api/2.0/alerts-internal/scheduled-insights", body={
            "parent_asset_name": f"users/{uid}",
            "scheduled_insight": {
                "display_name": display_name,
                "insight_type": "GENIE_CODE",
                "user_prompt": prompt,
            },
        })
        # Response may nest the insight; extract the full name for genie_task.configuration_id
        si = resp.get("scheduled_insight", resp)
        full_name = si.get("name", "")
        if not full_name:
            # Fallback: construct name from ID
            new_id = si.get("id", "")
            full_name = f"genie_code/users/{uid}/scheduled_insights/{new_id}" if new_id else ""
        if not full_name:
            print(f"  {auto_key}: WARNING could not extract name from response: {json.dumps(resp, indent=2)}")
        configuration_ids[auto_key] = full_name
        print(f"  {auto_key}: created automation {full_name}")

print(f"\nconfiguration_ids: {json.dumps(configuration_ids, indent=2)}")

In [0]:
def make_genie_task(task_key: str, config_id: str, params: dict, depends_on: list[str] | None = None, run_if: str | None = None) -> dict:
    """Build a genie_task dict for the jobs/reset payload.

    Uses 'parameters' (not 'context') — only 'parameters' supports {{input.*}}
    and {{job.*}} template variable resolution at runtime.
    """
    task = {
        "task_key": task_key,
        "genie_task": {
            "configuration_id": config_id,
            "parameters": {k: str(v) for k, v in params.items()},
        },
    }
    if depends_on:
        task["depends_on"] = [{"task_key": dk} for dk in depends_on]
    if run_if:
        task["run_if"] = run_if
    return task


def make_notebook_task(task_key: str, notebook_path: str, params: dict, depends_on: list[str] | None = None, run_if: str | None = None) -> dict:
    """Build a notebook_task dict (for the gate check)."""
    task = {
        "task_key": task_key,
        "notebook_task": {
            "notebook_path": notebook_path,
            "base_parameters": {k: str(v) for k, v in params.items()},
        },
    }
    if depends_on:
        task["depends_on"] = [{"task_key": dk} for dk in depends_on]
    if run_if:
        task["run_if"] = run_if
    return task


def generate_wave_tasks(waves: list[dict]) -> list[dict]:
    """Generate the full task graph from the manifest waves list."""
    tasks = []
    prev_gate_key = None

    for wave_cfg in waves:
        wid = wave_cfg["wave_id"]
        prefix = f"w{wid}"
        sessions = wave_cfg["sessions"]
        concurrency = wave_cfg.get("concurrency", len(sessions))

        # Common params injected into every Genie Code task
        common = {
            "run_id": "{{job.run_id}}",
            "branch": "{{job.parameters.branch}}",
            "build_root": "{{job.parameters.build_root}}",
            "manifest_path": "{{job.parameters.manifest_path}}",
        }

        # --- Sessions (for_each) ---
        # Include title in inputs so iterations are identifiable in the UI.
        inputs = json.dumps([
            {"session_id": s["session_id"], "title": s.get("title", s["session_id"])}
            for s in sessions
        ])
        session_params = {**common, "wave": wid, "session_id": "{{input.session_id}}"}

        inner_task = make_genie_task(
            task_key=f"{prefix}_session",
            config_id=configuration_ids["runner"],
            params=session_params,
        )
        inner_task["description"] = "{{input.session_id}}: {{input.title}}"

        sessions_task = {
            "task_key": f"{prefix}_sessions",
            "description": f"Wave {wid}: {', '.join(s.get('title', s['session_id']) for s in sessions)}",
            "for_each_task": {
                "inputs": inputs,
                "concurrency": concurrency,
                "task": inner_task,
            },
        }
        if prev_gate_key:
            # Dependency on a condition_task (If/else) requires an outcome.
            # "true" = only run when the gate passed.
            sessions_task["depends_on"] = [{"task_key": prev_gate_key, "outcome": "true"}]
        tasks.append(sessions_task)

        # --- Integrator ---
        integrate_params = {**common, "wave": wid}
        tasks.append(
            make_genie_task(
                task_key=f"{prefix}_integrate",
                config_id=configuration_ids["integrator"],
                params=integrate_params,
                depends_on=[f"{prefix}_sessions"],
                run_if="ALL_DONE",
            )
        )

        # --- Gate (deterministic notebook) ---
        gate_params = {
            "wave": wid,
            "run_id": "{{job.run_id}}",
            "build_root": "{{job.parameters.build_root}}",
            "manifest_path": "{{job.parameters.manifest_path}}",
        }
        tasks.append(
            make_notebook_task(
                task_key=f"{prefix}_gate",
                notebook_path=gate_notebook_path,
                params=gate_params,
                depends_on=[f"{prefix}_integrate"],
                run_if="ALL_DONE",
            )
        )

        # --- Gate check (If/else condition task) ---
        # Reads the gate notebook's taskValue. If false, all downstream tasks
        # are Excluded (zero-compute cascade stop via the exclusion cascade rule).
        tasks.append({
            "task_key": f"{prefix}_gate_check",
            "condition_task": {
                "op": "EQUAL_TO",
                "left": "{{" + f"tasks.{prefix}_gate.values.gate_passed" + "}}",
                "right": "true",
            },
            "depends_on": [{"task_key": f"{prefix}_gate"}],
        })

        prev_gate_key = f"{prefix}_gate_check"

    return tasks


# Generate from manifest
build_tasks = generate_wave_tasks(manifest["waves"])
print(f"Generated {len(build_tasks)} tasks across {len(manifest['waves'])} waves:")
for t in build_tasks:
    dep = [d["task_key"] for d in t.get("depends_on", [])]
    print(f"  {t['task_key']}" + (f" (depends: {dep})" if dep else ""))

In [0]:
def get_job(job_id: str) -> dict:
    return w.api_client.do("GET", "/api/2.1/jobs/get", query={"job_id": job_id})


def reset_job_tasks(job_id: str, new_tasks: list[dict]) -> None:
    """Replace a job's tasks entirely via jobs/reset, preserving all other settings."""
    job = get_job(job_id)
    settings = dict(job.get("settings", {}))
    settings["tasks"] = new_tasks
    # Remove fields that jobs/reset does not accept
    for drop_key in ["format", "deployment", "edit_mode", "source"]:
        settings.pop(drop_key, None)
    w.api_client.do("POST", "/api/2.1/jobs/reset", body={"job_id": int(job_id), "new_settings": settings})


# --- Reset the build job ---
reset_job_tasks(build_job_id, build_tasks)
print(f"Build job {build_job_id}: reset with {len(build_tasks)} tasks")

# --- Reset the smoke job ---
smoke_tasks = [
    {
        "task_key": "smoke_sessions",
        "for_each_task": {
            "inputs": json.dumps([{"session_id": "A"}, {"session_id": "B"}]),
            "concurrency": 2,
            "task": make_genie_task(
                task_key="smoke_session",
                config_id=configuration_ids["smoke"],
                params={"session_id": "{{input.session_id}}", "run_id": "{{job.run_id}}"},
            ),
        },
    }
]
reset_job_tasks(smoke_job_id, smoke_tasks)
print(f"Smoke job {smoke_job_id}: reset with {len(smoke_tasks)} tasks")

patched = {"build": len(build_tasks), "smoke": len(smoke_tasks)}

In [0]:
import time

smoke_result = None
if not run_smoke:
    print("run_smoke=false: skipping. Start the build job from the Jobs UI when ready.")
else:
    resp = w.api_client.do("POST", "/api/2.1/jobs/run-now", body={"job_id": int(smoke_job_id)})
    run_id_smoke = resp["run_id"]
    print(f"Started smoke run {run_id_smoke}: {w.config.host}/jobs/{smoke_job_id}/runs/{run_id_smoke}")

    deadline = time.time() + 30 * 60
    while True:
        run = w.api_client.do("GET", "/api/2.1/jobs/runs/get", query={"run_id": run_id_smoke})
        state = run.get("state", {})
        life = state.get("life_cycle_state", "")
        if life in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
            break
        if time.time() > deadline:
            print("WARN: smoke timed out after 30 min")
            break
        time.sleep(15)

    result_state = state.get("result_state", "UNKNOWN")
    print(f"Smoke run {run_id_smoke}: {result_state}")

    # Check iterations
    iterations = []
    for task in run.get("tasks", []):
        if task.get("for_each_task"):
            stats = task["for_each_task"].get("stats", {})
            iterations = task["for_each_task"].get("completed_iterations", [])
            print(f"  for_each stats: {stats}")
            print(f"  iterations: {len(iterations)}")

    smoke_result = {
        "run_id": run_id_smoke,
        "result_state": result_state,
        "iteration_count": len(iterations),
    }

In [0]:
result = {
    "build_job_id": build_job_id,
    "smoke_job_id": smoke_job_id,
    "skills_published": {k: (SKILLS[k]["fqn"] if v else False) for k, v in skills_published.items()},
    "configuration_ids": configuration_ids,
    "patched": patched,
    "smoke_result": smoke_result,
}
print(json.dumps(result, indent=2))
dbutils.notebook.exit(json.dumps(result))